# COMP5339 Project Assignment 2 - Electricity Sector Data Streaming and Analysis

### Run only this notebook file 'assignment2.ipynb'. 'visualization_task4.py' is integrated in this notebook !!!

## Installing required dependencies

In [26]:
pip install -r requirements.txt

'C:\Users\Anuj' is not recognized as an internal or external command,
operable program or batch file.


## Importing required libraries

In [8]:
import sys
import os
import time
import pathlib
import json
from datetime import datetime
import requests
import pandas as pd
import paho.mqtt.client as mqtt
import pandas as pd
import numpy as np
import subprocess
import webbrowser
import warnings

warnings.filterwarnings("ignore")

## Configurations

In [9]:
# Replaced the second argument with our valid Open Electricity API Key.
API_KEY = os.getenv("OE_API_KEY", "oe_3ZkU7EvuCfoNiQqE5t6NbiDj")
HEADERS = {"Authorization": f"Bearer {API_KEY}", "accept": "application/json"}

# --- Parameters ---
INTERVAL = "5m"
START = "2025-10-06T00:00:00"
END = "2025-10-13T00:00:00"
TOP_N_FACILITIES = 60  # Decided to focus only on top 60 facilities
FACILITY_METRICS = ["power", "emissions"] 
NETWORK_METRICS = ["price", "demand"]

# --- API Endpoints ---
NETWORK_BASE = "https://api.openelectricity.org.au/v4/market/network/NEM"
FACILITY_BASE = "https://api.openelectricity.org.au/v4/data/facilities/NEM" 
FACILITY_METADATA_BASE = "https://api.openelectricity.org.au/v4/facilities/"

# MQTT Configuration
MQTT_BROKER = "broker.hivemq.com"
MQTT_PORT = 1883
MQTT_TOPIC = "comp5339/electricity/data"
MQTT_CLIENT_ID = "publisher_task3"

# File Paths
CACHE_DIR = pathlib.Path("cache_openelectricity")
DATA_DIR = pathlib.Path("data")
PUBLISHED_LOG = DATA_DIR / "published_records.txt"
CONSOLIDATED_CSV = DATA_DIR / "consolidated_market_data.csv"

# Created directories to save files
CACHE_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

## Helper Functions for data retrieval and integration

### get_cached_path() - Generates the standardized cache file path for a facility.

In [10]:
def get_cached_path(code):
    # Uses START/END slices for simple date strings in the file name
    return CACHE_DIR / f"facility_{code}_5m_{START[:10]}_{END[:10]}.parquet"

### fetch_and_normalize_data() - Handles API request, error checking, and initial normalization. Returns raw data blocks for facilities or a DataFrame for network.

In [11]:
def fetch_and_normalize_data(base_url, query_params, context=""):
    try:
        resp = requests.get(base_url, params=query_params, headers=HEADERS, timeout=120)
        print(f"GET {context} -> {resp.status_code}")
        
        if resp.status_code >= 400:
            print(f"  URL: {resp.url}")
            print(f"  Response preview: {resp.text[:500]}")
            
            # If the API key is unauthorized/forbidden, terminate.
            if resp.status_code in [401, 403]:
                raise SystemExit(f"Critical API Request failed (Status {resp.status_code}): Check your API Key.")
                
            # For other non-critical errors (like 416 'No data available'), it returns empty data DataFrame for the caller to skip this facility.
            if "data/facilities" in base_url:
                # Explicitly handle 416 (Requested Range Not Satisfiable)
                if resp.status_code == 416:
                    print(f"  Warning: No data available for this range (416).")
                return [], True
            else:
                return pd.DataFrame(), False

        # Success path
        payload = resp.json()
        
        # Facility data uses the /data/facilities endpoint
        if "data/facilities" in base_url:
            # Returns raw data blocks for facility parsing later
            return payload.get("data", []), True
        
        # Network data uses the /market/network endpoint
        rows = []
        if isinstance(payload, dict) and isinstance(payload.get("data"), list):
            rows.append(pd.json_normalize(payload["data"]))
        elif isinstance(payload, list):
            rows.append(pd.json_normalize(payload))
        else:
            print("Warning: Payload format unexpected for network data.")
            return pd.DataFrame(), False 

        return pd.concat(rows, ignore_index=True, sort=False), False

    except requests.RequestException as e:
        # Catches network/connection errors
        raise SystemExit(f"Request failed: {e}. Ensure your API Key is valid and network connection is stable.")
    

### get_top_facilities() - Fetches facility metadata, filters by date, selects top N by capacity, and returns both the codes and a DataFrame with the static metadata.

In [12]:
def get_top_facilities(n, start_date, end_date):
    print("\nFetching facility metadata to determine top facilities...")
    
    # 1. Fetches all facilities metadata
    try:
        resp = requests.get(
            FACILITY_METADATA_BASE, 
            params={"network_id": "NEM"}, 
            headers=HEADERS, 
            timeout=60
        )
        resp.raise_for_status()
        fac_payload = resp.json()
    except requests.RequestException as e:
        print(f"Failed to fetch facility metadata: {e}")
        return [], pd.DataFrame() 

    fac_raw = pd.json_normalize(fac_payload.get("data", []))
    print(f"Retrieved {len(fac_raw)} facilities from NEM network")
    
    if fac_raw.empty:
        return [], pd.DataFrame()

    # 2. Extracts and process facility metadata (Explode units)
    u = fac_raw[["code", "units"]].explode("units", ignore_index=True)
    u = u.dropna(subset=["units"])
    unit_meta = pd.json_normalize(u["units"]).add_prefix("unit.")
    u = pd.concat([u.drop(columns=["units"]).reset_index(drop=True), unit_meta.reset_index(drop=True)], axis=1)

    # 3. Filters valid generators and by date range
    mask_valid = u["unit.dispatch_type"].isin(["GENERATOR", "BIDIRECTIONAL"])
    mask_excl = u["unit.fueltech_id"].isin(["battery_charging"]) # Excludes battery charging fuel technology
    u_ok = u[mask_valid & ~mask_excl].copy()
    
    start_dt = pd.to_datetime(start_date, utc=True)
    end_dt = pd.to_datetime(end_date, utc=True)
    
    # Parse timestamps
    u_ok["first_seen"] = pd.to_datetime(u_ok["unit.data_first_seen"].fillna(u_ok["unit.created_at"]), errors="coerce", utc=True)
    u_ok["last_seen"] = pd.to_datetime(u_ok["unit.data_last_seen"].fillna(u_ok["unit.updated_at"]), errors="coerce", utc=True)
    
    # Filters by date range
    u_ok = u_ok[(u_ok["last_seen"] >= start_dt) & (u_ok["first_seen"] <= end_dt)].copy()

    # Calculates facility capacity
    u_ok["unit.capacity_maximum"] = pd.to_numeric(u_ok["unit.capacity_maximum"], errors="coerce").fillna(0)

    # 4. Calculates facility capacity and select top 60
    u_ok["unit.capacity_maximum"] = pd.to_numeric(u_ok["unit.capacity_maximum"], errors="coerce").fillna(0)
    
    eligible_fac = (
        u_ok.groupby("code", as_index=False)
        .agg(capacity_max=("unit.capacity_maximum", "sum"))
        .sort_values("capacity_max", ascending=False)
        .reset_index(drop=True)
    )

    top_codes = eligible_fac["code"].head(n).tolist()
    
    # --- Start of requested Energy Type logic integration ---
    # Extracts and consolidate energy types per facility (User requested logic)
    facility_types = (
        u_ok.groupby("code")["unit.fueltech_id"]
        .apply(lambda x: ", ".join(sorted(set(x))))
        .reset_index()
        .rename(columns={"code": "facility_code", "unit.fueltech_id": "energy_type"})
    )
    # --- End of requested Energy Type logic integration ---
    
    # Filters the original facility metadata to include only the top 60 facilities
    eligible_meta_df = fac_raw[fac_raw["code"].isin(top_codes)].copy()

    # Renames the 'code' column to 'facility_code', before merging the energy types to prevent the 'ValueError'
    eligible_meta_df.rename(columns={'code': 'facility_code'}, inplace=True)
    
    # Merges the calculated energy types into the metadata directly on the common, now-renamed 'facility_code'
    eligible_meta_df = pd.merge(
        eligible_meta_df,
        facility_types,
        on="facility_code",
        how="left"
    )
    
    # --- Robust Column Selection and Renaming ---
    col_mapping = {
        "facility_code": "facility_code",
        "name": "name",
        "network_region": "network_region",
        "location.lat": "latitude",      
        "location.lng": "longitude",     
        "energy_type": "energy_type"  
    }
    
    api_cols_to_select = [api_col for api_col in col_mapping.keys() if api_col in eligible_meta_df.columns]
    
    # Ensures the newly calculated 'energy_type' is included for selection
    if 'energy_type' not in api_cols_to_select and 'energy_type' in eligible_meta_df.columns:
         api_cols_to_select.append('energy_type')
         
    eligible_meta_df = eligible_meta_df[api_cols_to_select].copy()
    rename_map = {api_col: final_col for api_col, final_col in col_mapping.items() if api_col in api_cols_to_select}
    eligible_meta_df.rename(columns=rename_map, inplace=True)
    
    required_final_cols = list(col_mapping.values())
    for col in required_final_cols:
        if col not in eligible_meta_df.columns:
            eligible_meta_df[col] = None
            
    eligible_meta_df = eligible_meta_df[required_final_cols]
    
    print(f"Found {len(eligible_fac)} active facilities in date range")
    print(f"Selecting top {n} facilities by capacity")
    
    return top_codes, eligible_meta_df

### parse_facility_blocks() - Parses data blocks from the /data/facilities endpoint and sums each metric (power/emissions) across all units per timestamp. 

In [13]:
def parse_facility_blocks(data_blocks, facility_code):    
    per_metric = {}
    
    for block in data_blocks:
        metric = block["metric"]
        frames = []
        for res in block.get("results", []):
            rows = res.get("data", [])
            if not rows:
                continue
            # Rows are [timestamp, value] pairs
            df = pd.DataFrame(rows, columns=["ts", metric])
            # Timezone conversion
            df["ts"] = pd.to_datetime(df["ts"], utc=True) 
            frames.append(df)
        
        if frames:
            mdf = pd.concat(frames, ignore_index=True)
            # Groups by timestamp and sum the metric value across all units
            mdf = mdf.groupby("ts", as_index=False).sum(numeric_only=True)
            per_metric[metric] = mdf
    
    if not per_metric:
        # Returns an empty DataFrame with expected columns if no data was found
        return pd.DataFrame(columns=["ts", "facility_code", "power", "emissions"])
    
    # Merges all metrics (power and emissions) on timestamp
    metrics = sorted(per_metric.keys())
    out = per_metric[metrics[0]].copy()
    for m in metrics[1:]:
        out = out.merge(per_metric[m], on="ts", how="outer")
    
    out.insert(1, "facility_code", facility_code)
    out.sort_values("ts", inplace=True)
    out.reset_index(drop=True, inplace=True)
    
    # Selects and reorder columns
    return out[[c for c in ["ts", "facility_code", "power", "emissions"] if c in out.columns]]

### parse_results_to_obj_market() - Safely converts string or series result columns back into Python objects.

In [14]:
def parse_results_to_obj_market(results_value):
    if results_value is None or pd.isna(results_value):
        return None
    
    if isinstance(results_value, (list, tuple, np.ndarray, pd.Series)):
        if len(results_value) == 0: return None
        results_value = results_value[0]

    if isinstance(results_value, (dict, list)):
        return results_value

    if isinstance(results_value, (str, bytes)):
        s = results_value.decode() if isinstance(results_value, bytes) else results_value
        s = s.strip()
        if not s: return None
        try: return json.loads(s)
        except Exception: pass
        try: return ast.literal_eval(s)
        except Exception: return None

    return None

### expand_network_results() - Expands a single row's 'results' column into multiple rows for the /market/network endpoint (price/demand).

In [15]:
def expand_network_results(row):
    results_obj = parse_results_to_obj_market(row.get("results"))
    if not results_obj:
        return pd.DataFrame()

    records = []
    metric = row.get("metric")

    if isinstance(results_obj, dict):
        results_obj = [results_obj]

    for series in results_obj:
        if not isinstance(series, dict):
            continue
        data_list = series.get("data", [])
        for item in data_list:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                ts, val = item[0], item[1]
                record = {
                    "time": ts,
                    "metric": metric,
                    "value": val,
                }
                records.append(record)

    return pd.DataFrame.from_records(records)

## 1. Data Retrieval (including caching)

In [17]:
# Dynamically determine the top 60 facility codes and their metadata
FACILITY_CODES, FACILITY_METADATA_DF = get_top_facilities(TOP_N_FACILITIES, START, END)


Fetching facility metadata to determine top facilities...
Retrieved 515 facilities from NEM network
Found 334 active facilities in date range
Selecting top 60 facilities by capacity


### 1a. Facility Data (Power and Emissions) - WITH CACHING

In [18]:
raw_facility_dfs = []
print(f"\nFetching power & emissions data for {len(FACILITY_CODES)} facilities...")
print("(Using cache in './cache_openelectricity' to avoid rate limits)")

for i, code in enumerate(FACILITY_CODES, 1):
    cache_file = get_cached_path(code)
    
    # 1. Checks if cached data exists
    if cache_file.exists():
        try:
            df_fac = pd.read_parquet(cache_file)
            print(f"[{i:02d}/{len(FACILITY_CODES)}] {code:15s} → using cached data ({len(df_fac)} rows)")
            raw_facility_dfs.append(df_fac)
            continue
        except Exception as e:
            print(f"[{i:02d}/{len(FACILITY_CODES)}] {code:15s} → Cache ERROR: {e}. Refetching.")
    
    # 2. Fetches from API if cache fails or is missing
    facility_params = [
        ("interval", INTERVAL),
        ("date_start", START),
        ("date_end", END),
        ("facility_code", code) 
    ]
    facility_params += [("metrics", m) for m in FACILITY_METRICS]

    # The fetch_and_normalize_data function now explicitly handles 416 (no data)
    data_blocks, _ = fetch_and_normalize_data(FACILITY_BASE, facility_params, context=f"facility ({code})")
    
    if data_blocks:
        df_fac = parse_facility_blocks(data_blocks, code)
        if not df_fac.empty:
            # 3. Cache the fetched data
            df_fac.to_parquet(cache_file, index=False)
            print(f"[{i:02d}/{len(FACILITY_CODES)}] {code:15s} → {len(df_fac):6d} rows → cached")
            raw_facility_dfs.append(df_fac)
        else:
            # This branch handles cases where API returned 200 but data block was empty, 
            # or the case where API returned 416 (handled inside fetch_and_normalize_data)
            print(f"[{i:02d}/{len(FACILITY_CODES)}] {code:15s} → No data found (Skipped)")

if not raw_facility_dfs:
    raise SystemExit("No facility data retrieved. Check API key and date range.")
    
# Combine all facilities.
combined_df_facility_raw = pd.concat(raw_facility_dfs, ignore_index=True)


Fetching power & emissions data for 60 facilities...
(Using cache in './cache_openelectricity' to avoid rate limits)
[01/60] TUMUT3          → using cached data (2016 rows)
[02/60] ERARING         → using cached data (2016 rows)
[03/60] BAYSW           → using cached data (2016 rows)
[04/60] LOYYANGA        → using cached data (2016 rows)
[05/60] GSTONE          → using cached data (2016 rows)
[06/60] WTAHB           → using cached data (2016 rows)
[07/60] MURRAY          → using cached data (2016 rows)
[08/60] TARONG          → using cached data (2016 rows)
[09/60] STANWELL        → using cached data (2016 rows)
[10/60] MP              → using cached data (2016 rows)
[11/60] YALLOURN        → using cached data (2016 rows)
[12/60] LOYYB           → using cached data (2016 rows)
[13/60] WIVENHOE        → using cached data (2016 rows)
GET facility (0LIDDELLBESS) -> 416
  URL: https://api.openelectricity.org.au/v4/data/facilities/NEM?interval=5m&date_start=2025-10-06T00%3A00%3A00&date_en

### 1b. Network Data (Price and Demand)

In [19]:
network_params = [
    ("interval", INTERVAL),
    ("date_start", START),
    ("date_end", END),
]
network_params += [("metrics", m) for m in NETWORK_METRICS]

# The API call returns a normalized DataFrame
combined_df_network_raw, _ = fetch_and_normalize_data(NETWORK_BASE, network_params, context="network")
combined_df_network_raw

GET network -> 200


,network_code,metric,unit,interval,date_start,date_end,groupings,results,network_timezone_offset
0,NEM,price,$/MWh,5m,2025-10-06T00:00:00+10:00,2025-10-12T23:55:00+10:00,[],"[{'name': 'price_total', 'date_start': '2025-1...",+10:00
1,NEM,demand,MW,5m,2025-10-06T00:00:00+10:00,2025-10-12T23:55:00+10:00,[],"[{'name': 'demand_total', 'date_start': '2025-...",+10:00


## 2. Data Integration and Materialization

### 2a. Prepare Long Facility Data (with Metadata)

In [20]:
long_df_facility = combined_df_facility_raw.rename(columns={"ts": "time"})
long_df_facility["time"] = long_df_facility["time"].dt.tz_localize(None) # Remove timezone for merge

# Merge facility time series with static metadata
final_long_facility_df = pd.merge(
    long_df_facility, 
    FACILITY_METADATA_DF, 
    on='facility_code', 
    how='left'
)

### 2b. Prepare Network Data (Price and Demand)

In [21]:
expanded_parts_network = [expand_network_results(r) for _, r in combined_df_network_raw.iterrows()]
long_df_network = pd.concat(expanded_parts_network, ignore_index=True)
long_df_network["time"] = pd.to_datetime(long_df_network["time"], utc=False).dt.tz_localize(None)

# Pivot network data: Columns are metric (e.g., price, demand)
market_data_wide = long_df_network.pivot_table( 
    index="time", 
    columns="metric", 
    values="value", 
    aggfunc="first"
).reset_index()
market_data_wide

metric,time,demand,price
0,2025-10-06 00:00:00,18860.88,16.594
1,2025-10-06 00:05:00,18723.58,20.520
2,2025-10-06 00:10:00,18514.75,16.536
3,2025-10-06 00:15:00,18537.81,20.512
4,2025-10-06 00:20:00,18478.41,16.444
...,...,...,...
2010,2025-10-12 23:35:00,20012.09,83.540
2011,2025-10-12 23:40:00,20099.43,96.254
2012,2025-10-12 23:45:00,19831.04,82.396
2013,2025-10-12 23:50:00,19816.86,86.556


### 2c. Final Merge and Cleanup 

In [22]:
final_merged_df = pd.merge(
    final_long_facility_df, 
    market_data_wide, 
    on='time', 
    how='left' 
)

# IMPUTATION: Fills missing price/demand values with the last known value (Forward Fill)
# Assigns the result of the ffill() operation directly back to the column.
final_merged_df['price'] = final_merged_df['price'].ffill()
final_merged_df['demand'] = final_merged_df['demand'].ffill()


# Sorting the final DataFrame: first by facility_code, then by time (ts)
final_merged_df = final_merged_df.sort_values(["facility_code", "time"]).reset_index(drop=True)

# Renames 'time' column back to 'ts' to match the user's image
final_merged_df.rename(columns={"time": "ts"}, inplace=True)

# Reorders columns to match the desired output structure
desired_order = [
    "facility_code", "ts", "power", "emissions", "name", 
    "network_region", "energy_type", "latitude", "longitude", 
    "price", "demand"
]
# Filters for columns that actually exist
existing_cols = [col for col in desired_order if col in final_merged_df.columns]
final_merged_df = final_merged_df[existing_cols]
final_merged_df

,facility_code,ts,power,emissions,name,network_region,energy_type,latitude,longitude,price,demand
0,0MREH,2025-10-05 14:00:00,0.0000,0.0000,Melbourne A1,VIC1,"battery, battery_discharging",-37.661274,144.726302,10.010,15887.63
1,0MREH,2025-10-05 14:05:00,0.0000,0.0000,Melbourne A1,VIC1,"battery, battery_discharging",-37.661274,144.726302,10.010,15887.63
2,0MREH,2025-10-05 14:10:00,0.0000,0.0000,Melbourne A1,VIC1,"battery, battery_discharging",-37.661274,144.726302,10.010,15887.63
3,0MREH,2025-10-05 14:15:00,2.3566,0.0000,Melbourne A1,VIC1,"battery, battery_discharging",-37.661274,144.726302,10.010,15887.63
4,0MREH,2025-10-05 14:20:00,0.0000,0.0000,Melbourne A1,VIC1,"battery, battery_discharging",-37.661274,144.726302,10.010,15887.63
...,...,...,...,...,...,...,...,...,...,...,...
118939,YALLOURN,2025-10-12 13:35:00,1082.7813,115.5389,Yallourn W,VIC1,coal_brown,-38.177596,146.347508,-6.490,15399.12
118940,YALLOURN,2025-10-12 13:40:00,1081.7188,115.4813,Yallourn W,VIC1,coal_brown,-38.177596,146.347508,-6.798,15455.95
118941,YALLOURN,2025-10-12 13:45:00,1073.8125,115.0529,Yallourn W,VIC1,coal_brown,-38.177596,146.347508,4.886,15714.93
118942,YALLOURN,2025-10-12 13:50:00,1081.6875,115.4796,Yallourn W,VIC1,coal_brown,-38.177596,146.347508,10.410,15880.19


### 2d. Materialization (CSV Caching)

In [23]:
print("Consolidated Dataset Generation Complete\n")
print("Combined DataFrame shape:", final_merged_df.shape)

# Save to the specified data folder
final_path = DATA_DIR / "consolidated_market_data.csv"
#final_merged_df.to_csv(final_path, index=False)
print(f"\nSaved consolidated data to: {final_path}")

Consolidated Dataset Generation Complete

Combined DataFrame shape: (118944, 11)

Saved consolidated data to: data\consolidated_market_data.csv


## 3. Data Publishing via MQTT

### 3a. Published records tracking

In [24]:
def load_published_records():    
    # Loads set of already published record IDs
    if PUBLISHED_LOG.exists():
        with open(PUBLISHED_LOG, 'r') as f:
            return set(line.strip() for line in f)
    return set()

def save_published_record(record_id):    
    # Saves published record ID to log
    DATA_DIR.mkdir(exist_ok=True) 
    with open(PUBLISHED_LOG, 'a') as f:
        f.write(f"{record_id}\n")

def publish_records(client, csv_path):    
    # Publish only records that haven't been published yet
    # Checks if the consolidated CSV exists
    if not csv_path.exists():
        print(f"\nError: Data file not found: {csv_path}.")
        return 0
        
    # Load data
    print(f"\nLoading data from: {csv_path}")
    df = pd.read_csv(csv_path)
    df['ts'] = pd.to_datetime(df['ts'])
    df = df.sort_values('ts').reset_index(drop=True)
    
    print(f"Loaded {len(df):,} records")
    
    # Create unique record ID
    df['record_id'] = df['facility_code'] + "_" + df['ts'].astype(str)
    
    # Filters to unpublished records
    published = load_published_records()
    new_records = df[~df['record_id'].isin(published)]
    
    if len(new_records) == 0:
        print("All records published — restarting replay from beginning.")
        if PUBLISHED_LOG.exists():
            PUBLISHED_LOG.unlink()  # deletes published log file
        published = set()
        new_records = df

    
    print(f"\nPublishing {len(new_records):,} NEW records...")
    print(f"\nSkipping {len(df) - len(new_records):,} already published")
    
    # Checks data quality
    with_coords = new_records['latitude'].notna().sum()
    with_price = new_records['price'].notna().sum() 
    with_demand = new_records['demand'].notna().sum() 
    
    print(f"   Records with coordinates: {with_coords:,}/{len(new_records):,} ({with_coords/len(new_records)*100:.1f}%)")
    print(f"   Records with price data: {with_price:,}/{len(new_records):,} ({with_price/len(new_records)*100:.1f}%)")
    print(f"   Records with demand data: {with_demand:,}/{len(new_records):,} ({with_demand/len(new_records)*100:.1f}%)")
    
    # Estimates time
    estimated_time = (len(new_records) * 0.1) / 60
    print(f"   Estimated time: {estimated_time:.1f} minutes")
    
    print("\n" + "-" * 80)
    
    published_count = 0
    start_time = time.time()
    
    last_values = {}  # Track last published values per facility

    for idx, row in new_records.iterrows():
        code = row['facility_code']
        power = float(row['power']) if pd.notna(row['power']) else 0.0
        emissions = float(row['emissions']) if pd.notna(row['emissions']) else 0.0
        price = float(row['price']) if pd.notna(row['price']) else None
        demand = float(row['demand']) if pd.notna(row['demand']) else None
    
        # Skip publishing if power and emissions haven't changed
        if code in last_values:
            prev = last_values[code]
            same_power = abs(prev['power'] - power) < 1e-6
            same_emissions = abs(prev['emissions'] - emissions) < 1e-6
    
            # 🔹 Only skip if BOTH are unchanged
            if same_power and same_emissions:
                continue
    
        # Create message with required fields
        message = {
            "timestamp": row['ts'].isoformat(),
            "facility_code": code,
            "facility_name": row['name'] if pd.notna(row['name']) else "Unknown",
            "network_region": row['network_region'] if pd.notna(row['network_region']) else "Unknown",
            "energy_type": row['energy_type'] if pd.notna(row['energy_type']) else "Unknown",
            "power": power,
            "emissions": emissions,
            "latitude": float(row['latitude']) if pd.notna(row['latitude']) else None,
            "longitude": float(row['longitude']) if pd.notna(row['longitude']) else None,
            "price": price,
            "demand": demand
        }
    
        # Publish to MQTT
        payload = json.dumps(message)
        result = client.publish(MQTT_TOPIC, payload, qos=1)
    
        if result.rc == mqtt.MQTT_ERR_SUCCESS:
            published_count += 1
            save_published_record(row['record_id'])
            last_values[code] = {"power": power, "emissions": emissions}
    
        # Progress indicator
        if published_count % 1000 == 0 and published_count > 0:
            elapsed = time.time() - start_time
            rate = published_count / elapsed
            remaining = (len(new_records) - published_count) / rate if rate > 0 else 0
            print(f"- {published_count:,}/{len(new_records):,} published "
                  f"({published_count/len(new_records)*100:.1f}%) "
                  f"| ETA: {remaining/60:.1f} min")
    
        # 0.1-second delay between messages
        time.sleep(0.1)
    
    elapsed_total = time.time() - start_time
    
    print("-" * 80)
    print(f"\nPublished {published_count:,} messages in {elapsed_total/60:.1f} minutes")
    return published_count

### 3b. MQTT callbacks

In [25]:
def on_connect(client, userdata, flags, reason_code, properties=None, *args):
    if reason_code == 0:
        print(f"Connected to MQTT broker at {datetime.now().strftime('%H:%M:%S')}")
    else:
        print(f"Connection failed with code {reason_code}")

def on_disconnect(client, userdata, disconnect_reason_code, properties=None, *args):
    if disconnect_reason_code != 0:
        print(f"Unexpected disconnection: {disconnect_reason_code}")

## 4. Data Subscription and Visualisation (including continuous execution)

#### To maintain continuous execution, publishing and subscription are done simultaneously

#### Main function for Task 5: Continuous Execution. The publishing cycle runs every 60 seconds, relying on an existing CSV.

#### The function 'run_continuous_data_pipeline()' publishes messages to the broker and calls 'visualization_task4.py' which includes subscription and dashboard visualization

#### To quite the execution, click the below cell and  press 'I' two times !!

In [6]:
def run_continuous_data_pipeline():    
    client = mqtt.Client(
        client_id=MQTT_CLIENT_ID,
        callback_api_version=mqtt.CallbackAPIVersion.VERSION2) 
    
    client.on_connect = on_connect
    client.on_disconnect = on_disconnect
    
    dashboard_process = None 
    
    DASHBOARD_FILE = "visualization_task4.py" 
    DASHBOARD_URL = 'http://127.0.0.1:8050/'
    DASHBOARD_LOG = DATA_DIR / "dash_error_log.txt"

    # Clean up old log file before starting
    if DASHBOARD_LOG.exists():
        try:
            DASHBOARD_LOG.unlink()
        except OSError as e:
            print(f"Warning: Could not remove old log file {DASHARD_LOG}: {e}")
            
    try:
        print(f"\nConnecting to MQTT broker: {MQTT_BROKER}:{MQTT_PORT}")
        client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)
        client.loop_start()
        
        # --- LAUNCH DASHBOARD SUBPROCESS ---
        
        try:
            print(f"Attempting to launch dashboard file: {DASHBOARD_FILE} in a separate background process.")
            
            with open(DASHBOARD_LOG, "w") as log_file:
                # Use sys.executable for cross-platform compatibility
                dashboard_process = subprocess.Popen([sys.executable, DASHBOARD_FILE], 
                                                     stdout=subprocess.DEVNULL,
                                                     stderr=log_file)
            
            print(f"Dashboard process started (PID: {dashboard_process.pid}).")
            
            MAX_RETRIES = 15
            RETRY_DELAY = 1 
            server_ready = False
            
            print(f"Polling {DASHBOARD_URL} for server readiness (Max wait: {MAX_RETRIES * RETRY_DELAY}s)...")
            
            for i in range(MAX_RETRIES):
                if dashboard_process.poll() is not None:
                    print(f"Dash process terminated unexpectedly after {i} checks. Status code: {dashboard_process.returncode}")
                    if DASHBOARD_LOG.exists():
                        print("-" * 35 + f" Captured Error Log from {DASHBOARD_FILE} " + "-" * 35)
                        with open(DASHBOARD_LOG, "r") as f:
                            error_content = f.read()
                            if error_content: print(error_content.strip())
                            else: print("Log file exists but is empty. Possible non-Python execution failure.")
                        print("-" * 100)
                    else: print("No error log file found.")
                    break

                try:
                    response = requests.get(DASHBOARD_URL, timeout=1)
                    if response.status_code == 200:
                        server_ready = True
                        print(f"Server responded with status 200 after {i+1} attempts.")
                        break
                except requests.exceptions.RequestException:
                    pass
                    
                time.sleep(RETRY_DELAY)

            if server_ready:
                webbrowser.open_new_tab(DASHBOARD_URL)
                print("Browser tab launched successfully.")
            else:
                if dashboard_process.poll() is not None:
                    print(f"Error: Dash server failed to start or did not respond at {DASHBOARD_URL}.")
                    print(f"         The dashboard process terminated prematurely. **Check the log file for details.**")
                else:
                    print(f"Error: Dash server did not respond at {DASHBOARD_URL} within {MAX_RETRIES} seconds.")
                    print("         The process is still running, but the server is unreachable. Check port conflicts.")


        except FileNotFoundError:
            print(f"Warning: Could not find 'python' executable or the file {DASHBOARD_FILE}.")
            print("         The dashboard must be started manually.")
        except Exception as e:
            print(f"Warning: Failed to start dashboard process or open browser: {e}")
        # --- END LAUNCH DASHBOARD SUBPROCESS ---
        
        # --- The main continuous loop ---
        round_number = 1
        
        while True:
            print("\n" + "=" * 80)
            print(f"ROUND {round_number} - Starting data retrieval and publishing")
            print(f"   Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("=" * 80)
            
            # 1. PUBLISHING (Task 3)
            # This publishes any newly added records from the consolidated CSV file
            published_count = publish_records(client, CONSOLIDATED_CSV)
            
            print("\n" + "=" * 80)
            print(f"ROUND {round_number} Complete")
            print(f"   Published: {published_count:,} new records")
            print(f"   Total published (all time): {len(load_published_records()):,}")
            print("=" * 80)
            
            # CRITICAL: 60-second delay between rounds (Task 5 requirement)
            print(f"\nWaiting 60 seconds before next round...")
            print(f"   Next round will start at: {datetime.fromtimestamp(time.time() + 60).strftime('%H:%M:%S')}")
            
            time.sleep(60)  # Mandatory 60-second delay
            
            round_number += 1
            
    except KeyboardInterrupt:
        print("\n\nContinuous Pipeline Interrupted by user (Pressed 'I' 2 times)")
    except Exception as e:
        print(f"\n✗ Critical Error in Pipeline: {e}")
        import traceback
        traceback.print_exc()
    finally:
        client.loop_stop()
        client.disconnect()
        print("\nMQTT Client Disconnected. Pipeline Terminated.")

        # --- CLEANUP DASHBOARD PROCESS ---
        if dashboard_process and dashboard_process.poll() is None:
            print("Terminating dashboard process...")
            dashboard_process.terminate()
            time.sleep(1) 
            if dashboard_process.poll() is None:
                 dashboard_process.kill()
            print("Dashboard process terminated.")
        # --- END CLEANUP DASHBOARD PROCESS ---

# --- EXECUTION CELL ---
if __name__ == '__main__':
    run_continuous_data_pipeline()


Connecting to MQTT broker: broker.hivemq.com:1883
Attempting to launch dashboard file: visualization_task4.py in a separate background process.
Dashboard process started (PID: 19372).
Polling http://127.0.0.1:8050/ for server readiness (Max wait: 15s)...
Connected to MQTT broker at 15:21:59
Server responded with status 200 after 3 attempts.
Browser tab launched successfully.

ROUND 1 - Starting data retrieval and publishing
   Time: 2025-11-09 15:22:03

Loading data from: data\consolidated_market_data.csv
Loaded 118,944 records

Publishing 104,772 NEW records...

Skipping 14,172 already published
   Records with coordinates: 104,772/104,772 (100.0%)
   Records with price data: 104,772/104,772 (100.0%)
   Records with demand data: 104,772/104,772 (100.0%)
   Estimated time: 174.6 minutes

--------------------------------------------------------------------------------
- 1,000/104,772 published (1.0%) | ETA: 176.8 min
- 2,000/104,772 published (1.9%) | ETA: 174.9 min
- 3,000/104,772 pub